# t-SNE Visualization of Good vs. Poor Segmentation Cases

Projects the radiomic feature vectors into 2D using t-SNE to visualize the separation between concordant-good and concordant-poor segmentation cases.


In [ ]:
import pandas as pd
import pickle as pkl
from sklearn.manifold import TSNE
import torch
import numpy as np


In [ ]:
def load_unet_result(path, _print):
    unet_df = pd.read_csv(path, index_col = 'Unnamed: 0')
    index_values = []
    for _index in unet_df.index:
        index_values.append(_index.split('-seg')[0])

    unet_df.index = index_values  
#     unet_df.drop(['WT jaccard', 'TC jaccard', 'ET jaccard'], axis = 1, inplace = True)
    summary_unet_df = pd.DataFrame(zip(unet_df.mean().values.tolist(), 
                                       unet_df.std().values.tolist()), 
                                   columns = ['mean', 'std'], index = unet_df.columns)
    if _print:
        print("****UNet******")
        print(summary_unet_df)
    return summary_unet_df, unet_df


In [ ]:
_print = True
path = '../Results/Result/Vanilla_Unet/Unet_test_dice.csv'
summary_unet_df, unet_df = load_unet_result(path, _print)


In [ ]:
unet_df[unet_df['WT dice'] > 0.9][0:20]


In [ ]:
unet_df[unet_df['WT dice'] < 0.8][0:10]


In [ ]:
path = '../Results/Analysis_Results/t-sne/Enc4_numpy.pkl'
Enc_features = torch.load(path, map_location=torch.device('cpu'))


In [ ]:
with open('../Results/Analysis_Results/t-sne/Enc2_numpy.pkl', 'rb') as f:
    Enc_features = pkl.load(f)


In [ ]:
all_features = []
all_labels = []
count = 0
for p_id in Enc_features.keys():
    if (unet_df.loc[p_id, 'WT dice'] <.98):
        continue
    features = Enc_features[p_id]['feature']
    mask_resized = Enc_features[p_id]['resized_mask']
    
    features_flattened = features.view(192, -1).T  # Shape: [num_voxels, 192]
    mask_flat = mask_resized.view(-1)

#     tumor_features = features_flattened[mask_flat != 0]
#     tumor_labels = mask_flat[mask_flat != 0]
    
    # Label as 'tumor' (1) and 'non-tumor' (0)
    labels = (mask_flat != 0).long()  # Convert to 0 and 1
    
    # Append to the aggregated lists
    all_features.append(features_flattened)
    all_labels.append(labels)
    count += 2
    if count > 4:
        break
len(all_features)


In [ ]:
# Concatenate all features and labels
all_features_combined = torch.cat(all_features, dim=0)
all_labels_combined = torch.cat(all_labels, dim=0)
all_features_combined.shape


In [ ]:
features_numpy = all_features_combined.numpy()
labels_numpy = all_labels_combined.numpy()


In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

tsne = TSNE(n_components=2, # perplexity controls local vs. global structure trade-off (typically 5–50)
    perplexity=50, n_iter=5000, learning_rate='auto', init='pca')
tsne_results = tsne.fit_transform(features_numpy)



In [ ]:
colors = ['red' if label == 1 else 'blue' for label in labels_numpy]
# Plotting without outliers
plt.figure(figsize=(10, 8))
plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=colors)
plt.xlabel('TSNE Component 1')
plt.ylabel('TSNE Component 2')
plt.title('t-SNE Visualization of Tumor vs. Non-Tumor Regions (Outliers Removed)')
plt.show()
